# Load dataset

In [2]:
import ast
import time

In [3]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

d:\work\cyshield_Task\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
DATA_PATH = "../data/raw/1k_stories_100_genre.csv"
BENCHMARK_PATH = "../benchmark/task1_benchmark_v1.csv"

df = pd.read_csv(DATA_PATH)
benchmark = pd.read_csv(BENCHMARK_PATH)

print("Dataset:", df.shape)
print("Benchmark:", benchmark.shape)

display(df.head())
display(benchmark.head())

Dataset: (1000, 4)
Benchmark: (50, 12)


,id,title,story,genre
0,457580,The Chronicles of the Cosmic Rift,"In the year 2250, Earth had made significant s...",Science Fiction
1,297904,Eldoria's Enchanted Whispers,"In a land far away, where the sun shone bright...",Fantasy
2,620436,Echoes of Whispered Shadows,"Once upon a time, in a small, tranquil town ca...",Mystery
3,634687,Emerald Amulet Chronicles Revealed,"Once upon a time in the 16th century, a small ...",Historical Adventure
4,513427,The Shadows of St. Augustine,In the sun-drenched coastal city of St. August...,Thriller


,query_id,query,query_type,difficulty,relevant_story_ids,expected_answer,ground_truth,notes,relevant_story_titles,relevant_story_genres,relevant_story_word_counts,num_relevant_stories
0,Q001,What is the title of story 457580?,exact_lookup,easy,[457580],The Chronicles of the Cosmic Rift,gold,NaN,"[""The Chronicles of the Cosmic Rift""]","[""Science Fiction""]",[1309],1
1,Q002,What genre is the story with ID 297904?,exact_lookup,easy,[297904],Fantasy,gold,NaN,"[""Eldoria's Enchanted Whispers""]","[""Fantasy""]",[1081],1
2,Q003,"What is the title of the story ""Neon Nights""?",exact_title,easy,[352800],Neon Nights,gold,NaN,"[""Neon Nights""]","[""Cyberpunk""]",[2467],1
3,Q004,"What genre is ""The Sands of Time""?",exact_title,easy,[204942],Historical Fiction,gold,NaN,"[""The Sands of Time""]","[""Historical Fiction""]",[2318],1
4,Q005,"Give me the ID of ""The Enigma of Elmwood Manor"".",exact_title,easy,[111926],111926,gold,NaN,"[""The Enigma of Elmwood Manor""]","[""Mystery""]",[1430],1


# Load baseline embedding model

In [5]:
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

model = SentenceTransformer(MODEL_NAME)

print("Max sequence length:", model.max_seq_length)
print("Embedding dimension:", model.get_sentence_embedding_dimension())

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2443.42it/s]


Max sequence length: 256
Embedding dimension: 384


C:\Users\shawk\AppData\Local\Temp\ipykernel_19508\4241072960.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", model.get_sentence_embedding_dimension())


In [35]:
tokenizer = model.tokenizer

token_lengths = df["story"].apply(
    lambda text: len(
        tokenizer(
            text,
            add_special_tokens=True,
            truncation=False
        )["input_ids"]
    )
)

df["token_count_minilm"] = token_lengths

df["token_count_minilm"].describe(
    percentiles=[0.5, 0.9, 0.95, 0.99]
)

count    1000.000000
mean     1203.003000
std       515.297245
min        95.000000
50%      1193.000000
90%      1736.500000
95%      2002.250000
99%      2720.690000
max      3670.000000
Name: token_count_minilm, dtype: float64

In [36]:
max_len = model.max_seq_length

truncated_count = (df["token_count_minilm"] > max_len).sum()
truncated_pct = truncated_count / len(df) * 100

print(f"Model max length: {max_len}")
print(f"Stories exceeding limit: {truncated_count}/{len(df)}")
print(f"Percentage truncated: {truncated_pct:.2f}%")

Model max length: 256
Stories exceeding limit: 902/1000
Percentage truncated: 90.20%


# Embedding the stories

In [37]:
stories = df["story"].tolist()

start = time.perf_counter()

story_embeddings = model.encode(
    stories,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

embedding_time = time.perf_counter() - start

print("Embeddings shape:", story_embeddings.shape)
print(f"Total embedding time: {embedding_time:.2f}s")
print(f"Average per story: {embedding_time / len(df) * 1000:.2f} ms")

Batches: 100%|██████████| 32/32 [00:32<00:00,  1.01s/it]

Embeddings shape: (1000, 384)
Total embedding time: 32.45s
Average per story: 32.45 ms


In [38]:
def retrieve(query, top_k=10):
    query_embedding = model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True
    )[0]

    scores = story_embeddings @ query_embedding

    top_indices = np.argsort(scores)[::-1][:top_k]

    results = df.iloc[top_indices][
        ["id", "title", "genre"]
    ].copy()

    results["score"] = scores[top_indices]

    return results

In [39]:
retrieve(
    "Find stories involving time travel.",
    top_k=5
)

,id,title,genre,score
723,523159,The Time Traveling Chronicles of Tom and Sarah,Time Travel,0.596287
523,687889,The Time Travelers Chronicles: The Adventures ...,Time Travel,0.538949
922,390933,The Timeless Clock,Alternate History,0.522692
374,161200,The Timeless Tales of Tom,Time-Loop,0.515624
655,441294,The Time Travelers Quest,Soft Science Fiction,0.502637


In [40]:
def parse_relevant_ids(value):
    if isinstance(value, list):
        return value

    return ast.literal_eval(value)

In [41]:
benchmark[["query", "relevant_story_ids"]].head()

,query,relevant_story_ids
0,What is the title of story 457580?,[457580]
1,What genre is the story with ID 297904?,[297904]
2,"What is the title of the story ""Neon Nights""?",[352800]
3,"What genre is ""The Sands of Time""?",[204942]
4,"Give me the ID of ""The Enigma of Elmwood Manor"".",[111926]


In [42]:
parse_relevant_ids(
    benchmark.iloc[0]["relevant_story_ids"]
)

[457580]

# Evaluation

In [43]:
def get_retrieved_ids(query, top_k=10):
    results = retrieve(query, top_k=top_k)
    return results["id"].tolist()

In [44]:
def recall_at_k(retrieved, relevant, k):
    relevant = set(relevant)
    retrieved = set(retrieved[:k])

    if len(relevant) == 0:
        return 0.0

    return len(retrieved & relevant) / len(relevant)


def hit_at_k(retrieved, relevant, k):
    relevant = set(relevant)
    retrieved = set(retrieved[:k])

    return float(len(retrieved & relevant) > 0)


def reciprocal_rank(retrieved, relevant):
    relevant = set(relevant)

    for rank, story_id in enumerate(retrieved, start=1):
        if story_id in relevant:
            return 1.0 / rank

    return 0.0

## Run benchmark

In [45]:
rows = []

for _, row in benchmark.iterrows():

    query = row["query"]
    relevant = parse_relevant_ids(row["relevant_story_ids"])

    start = time.perf_counter()

    retrieved = get_retrieved_ids(
        query,
        top_k=10
    )

    latency = time.perf_counter() - start

    rows.append({
        "query_id": row["query_id"],
        "query_type": row["query_type"],
        "query": query,

        "relevant_ids": relevant,
        "retrieved_ids": retrieved,

        "recall@1": recall_at_k(retrieved, relevant, 1),
        "recall@3": recall_at_k(retrieved, relevant, 3),
        "recall@5": recall_at_k(retrieved, relevant, 5),
        "recall@10": recall_at_k(retrieved, relevant, 10),

        "hit@1": hit_at_k(retrieved, relevant, 1),
        "hit@3": hit_at_k(retrieved, relevant, 3),
        "hit@5": hit_at_k(retrieved, relevant, 5),
        "hit@10": hit_at_k(retrieved, relevant, 10),

        "rr": reciprocal_rank(retrieved, relevant),

        "latency_ms": latency * 1000
    })


results = pd.DataFrame(rows)

results.head()

,query_id,query_type,query,relevant_ids,retrieved_ids,recall@1,recall@3,recall@5,recall@10,hit@1,hit@3,hit@5,hit@10,rr,latency_ms
0,Q001,exact_lookup,What is the title of story 457580?,[457580],"[889582, 382840, 216737, 335949, 891965, 61067...",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,31.7124
1,Q002,exact_lookup,What genre is the story with ID 297904?,[297904],"[308506, 928579, 319739, 574317, 889582, 69466...",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,22.2028
2,Q003,exact_title,"What is the title of the story ""Neon Nights""?",[352800],"[710800, 840003, 756026, 228442, 738701, 83602...",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,20.3219
3,Q004,exact_title,"What genre is ""The Sands of Time""?",[204942],"[81749, 68052, 390933, 161200, 584500, 77833, ...",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,21.8937
4,Q005,exact_title,"Give me the ID of ""The Enigma of Elmwood Manor"".",[111926],"[533303, 111926, 891965, 361939, 928789, 83390...",0.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,0.5,16.9863


In [46]:
metric_columns = [
    "recall@1",
    "recall@3",
    "recall@5",
    "recall@10",
    "hit@1",
    "hit@3",
    "hit@5",
    "hit@10",
    "rr",
    "latency_ms"
]

overall = results[metric_columns].mean()

overall

recall@1       0.21400
recall@3       0.29400
recall@5       0.38200
recall@10      0.41400
hit@1          0.34000
hit@3          0.46000
hit@5          0.58000
hit@10         0.58000
rr             0.41800
latency_ms    17.13504
dtype: float64

In [20]:
print("=== Naive Full-Story Retrieval Baseline ===")

print(f"Recall@1 : {results['recall@1'].mean():.4f}")
print(f"Recall@3 : {results['recall@3'].mean():.4f}")
print(f"Recall@5 : {results['recall@5'].mean():.4f}")
print(f"Recall@10: {results['recall@10'].mean():.4f}")

print()

print(f"Hit@1    : {results['hit@1'].mean():.4f}")
print(f"Hit@3    : {results['hit@3'].mean():.4f}")
print(f"Hit@5    : {results['hit@5'].mean():.4f}")
print(f"Hit@10   : {results['hit@10'].mean():.4f}")

print()

print(f"MRR       : {results['rr'].mean():.4f}")
print(f"Avg latency: {results['latency_ms'].mean():.2f} ms")

=== Naive Full-Story Retrieval Baseline ===
Recall@1 : 0.2140
Recall@3 : 0.2940
Recall@5 : 0.3820
Recall@10: 0.4140

Hit@1    : 0.3400
Hit@3    : 0.4600
Hit@5    : 0.5800
Hit@10   : 0.5800

MRR       : 0.4180
Avg latency: 20.50 ms


In [21]:
by_type = (
    results
    .groupby("query_type")
    [
        [
            "recall@1",
            "recall@3",
            "recall@5",
            "recall@10",
            "hit@1",
            "hit@5",
            "hit@10",
            "rr"
        ]
    ]
    .mean()
    .round(3)
)

by_type

,recall@1,recall@3,recall@5,recall@10,hit@1,hit@5,hit@10,rr
query_type,,,,,,,,
content_qa,0.467,0.533,0.733,0.733,0.467,0.733,0.733,0.547
cross_story,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
exact_lookup,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
exact_title,0.000,0.200,0.200,0.200,0.000,0.200,0.200,0.100
long_story_qa,0.600,0.800,0.800,0.800,0.600,0.800,0.800,0.667
metadata,0.062,0.125,0.200,0.275,0.625,0.750,0.750,0.667
semantic_discovery,0.020,0.070,0.150,0.250,0.200,0.700,0.700,0.353


# Inspect failures

In [22]:
failures = results[
    results["hit@10"] == 0
][
    [
        "query_id",
        "query_type",
        "query",
        "relevant_ids",
        "retrieved_ids"
    ]
]

failures.head()

,query_id,query_type,query,relevant_ids,retrieved_ids
0,Q001,exact_lookup,What is the title of story 457580?,[457580],"[889582, 382840, 216737, 335949, 891965, 61067..."
1,Q002,exact_lookup,What genre is the story with ID 297904?,[297904],"[308506, 928579, 319739, 574317, 889582, 69466..."
2,Q003,exact_title,"What is the title of the story ""Neon Nights""?",[352800],"[710800, 840003, 756026, 228442, 738701, 83602..."
3,Q004,exact_title,"What genre is ""The Sands of Time""?",[204942],"[81749, 68052, 390933, 161200, 584500, 77833, ..."
5,Q006,exact_title,"Which story is titled ""Stars of Tomorrow"" and ...",[798586],"[710800, 125976, 928579, 5816, 91114, 308506, ..."


In [23]:
len(failures)

21

In [24]:
failure = failures.iloc[0]

print("Query:")
print(failure["query"])

print("\nExpected:")
display(
    df[df["id"].isin(failure["relevant_ids"])][
        ["id", "title", "genre", "token_count_minilm"]
    ]
)

print("\nRetrieved:")
display(
    retrieve(
        failure["query"],
        top_k=10
    )
)

Query:
What is the title of story 457580?

Expected:


,id,title,genre,token_count_minilm
0,457580,The Chronicles of the Cosmic Rift,Science Fiction,1661



Retrieved:


,id,title,genre,score
502,889582,The Secret of the Lost Locket,Mystery,0.445807
387,382840,The Shadows of Silverwood,Mystery Thriller,0.441261
638,216737,Echoes of Fate,Psychological Thriller,0.433446
434,335949,Justice for All,Legal Drama,0.423910
604,891965,The Shadows of Darkness,Thriller,0.413494
658,610670,Tales from the Neighborhood,Anthology,0.410703
474,45171,The Time-Loop Chronicles: The Battle for Tomorrow,Time-Loop,0.401600
342,869376,The Unyielding Soul,Inspirational,0.396686
366,811674,A New Dawn at 40,Coming-of-Middle-Age,0.395795
732,919144,Shadows of the Night,Noir,0.394318


Of course its a failure as it only depends on semantic search not exact lookup

## Naive full-story embedding using MiniLM is unsuitable as a document representation because 90.2% of stories exceed the model's maximum sequence length.

# ------------------------------------------------------------------------------------------------------------------------

# Experiment2 -> chunking and embed each chunk and agg them into one vector per story

In [47]:
CHUNK_SIZE = model.max_seq_length - 2
 
print(f"Chunk size: {CHUNK_SIZE} tokens")

Chunk size: 254 tokens


In [48]:
def chunk_text(text, tokenizer, chunk_size=256):
    token_ids = tokenizer.encode(
        text,
        add_special_tokens=False
    )

    chunks = []

    for i in range(0, len(token_ids), chunk_size):
        chunk_ids = token_ids[i:i + chunk_size]

        chunk_text = tokenizer.decode(
            chunk_ids,
            skip_special_tokens=True
        )

        chunks.append(chunk_text)

    return chunks

In [49]:
longest_idx = df["token_count_minilm"].idxmax()

story = df.loc[longest_idx, "story"]

chunks = chunk_text(
    story,
    tokenizer,
    CHUNK_SIZE
)

print("Story tokens:", df.loc[longest_idx, "token_count_minilm"])
print("Number of chunks:", len(chunks))

for i, chunk in enumerate(chunks[:3]):
    print(f"\n--- Chunk {i + 1} ---")
    print(chunk[:300])

Story tokens: 3670
Number of chunks: 15

--- Chunk 1 ---
in the sleepy town of elmswood, life seemed to move at a leisurely pace. the residents went about their daily routines with little to no excitement, making the occasional small talk while waiting in line at the grocery store or sharing a friendly nod as they passed each other on the street. elmswood

--- Chunk 2 ---
reflection of his personality, warm and inviting, with walls adorned with diplomas and certificates that attested to his expertise in the field of psychiatry. as the days went by, dr. reynolds found himself growing increasingly restless. his patients began to sense his unease, and it wasn ' t long b

--- Chunk 3 ---
that they had kept hidden away for years. it was as if they had discovered a shared understanding of the world, a connection that neither could explain. but as their relationship deepened, so too did the darkness that lay hidden within dr. reynolds ' s mind. his dreams were plagued by a recurring ni


In [50]:
df["num_chunks"] = df["story"].apply(
    lambda text: len(
        chunk_text(
            text,
            tokenizer,
            CHUNK_SIZE
        )
    )
)

df["num_chunks"].describe(
    percentiles=[0.5, 0.9, 0.95, 0.99]
)

count    1000.000000
mean        5.223000
std         2.087493
min         1.000000
50%         5.000000
90%         7.000000
95%         8.000000
99%        11.000000
max        15.000000
Name: num_chunks, dtype: float64

In [51]:
print("Total stories:", len(df))
print("Total chunks:", df["num_chunks"].sum())
print("Average chunks/story:", df["num_chunks"].mean())
print("Max chunks/story:", df["num_chunks"].max())

Total stories: 1000
Total chunks: 5223
Average chunks/story: 5.223
Max chunks/story: 15


In [52]:
def encode_story_with_mean_pooling(
    text,
    model,
    tokenizer,
    chunk_size=256
):
    chunks = chunk_text(
        text,
        tokenizer,
        chunk_size
    )

    chunk_embeddings = model.encode(
        chunks,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    story_embedding = chunk_embeddings.mean(axis=0)

    # Re-normalize after mean pooling
    story_embedding = story_embedding / np.linalg.norm(
        story_embedding
    )

    return story_embedding

# Sanity Checking

In [53]:
test_embedding = encode_story_with_mean_pooling(
    df.iloc[0]["story"],
    model,
    tokenizer,
    CHUNK_SIZE
)

print(test_embedding.shape)
print(np.linalg.norm(test_embedding))

(384,)
1.0


# Encode all the stories

In [54]:
chunk_pooled_embeddings = []

start = time.perf_counter()

for i, story in enumerate(df["story"]):

    embedding = encode_story_with_mean_pooling(
        story,
        model,
        tokenizer,
        CHUNK_SIZE
    )

    chunk_pooled_embeddings.append(embedding)

    if (i + 1) % 100 == 0:
        print(f"Processed {i + 1}/{len(df)} stories")


chunk_pooled_embeddings = np.vstack(
    chunk_pooled_embeddings
)

chunk_embedding_time = time.perf_counter() - start

print()
print("Shape:", chunk_pooled_embeddings.shape)
print(f"Total indexing time: {chunk_embedding_time:.2f}s")
print(
    f"Average per story: "
    f"{chunk_embedding_time / len(df) * 1000:.2f} ms"
)

Processed 100/1000 stories
Processed 200/1000 stories
Processed 300/1000 stories
Processed 400/1000 stories
Processed 500/1000 stories
Processed 600/1000 stories
Processed 700/1000 stories
Processed 800/1000 stories
Processed 900/1000 stories
Processed 1000/1000 stories

Shape: (1000, 384)
Total indexing time: 185.56s
Average per story: 185.56 ms


## New Retrival function

In [55]:
def retrieve_chunk_pooled(query, top_k=10):

    query_embedding = model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True
    )[0]

    scores = chunk_pooled_embeddings @ query_embedding

    top_indices = np.argsort(scores)[::-1][:top_k]

    results = df.iloc[top_indices][
        ["id", "title", "genre"]
    ].copy()

    results["score"] = scores[top_indices]

    return results

In [56]:
retrieve_chunk_pooled(
    "Find stories involving time travel.",
    top_k=5
)


,id,title,genre,score
523,687889,The Time Travelers Chronicles: The Adventures ...,Time Travel,0.533591
922,390933,The Timeless Clock,Alternate History,0.522692
874,939455,The Time-Loop Chronicles: Echoes of Fate,Time-Loop,0.494765
123,461467,The Time Travel Chronicles of the Lost Timekee...,Time Travel,0.493246
323,764469,The Chronicles of Time: The Relic of the Ages,Time Travel,0.491489


### Benchmarking

In [57]:
rows = []

for _, row in benchmark.iterrows():

    query = row["query"]

    relevant = parse_relevant_ids(
        row["relevant_story_ids"]
    )

    start = time.perf_counter()

    retrieved_df = retrieve_chunk_pooled(
        query,
        top_k=10
    )

    latency = time.perf_counter() - start

    retrieved = retrieved_df["id"].tolist()

    rows.append({
        "query_id": row["query_id"],
        "query_type": row["query_type"],
        "query": query,

        "relevant_ids": relevant,
        "retrieved_ids": retrieved,

        "recall@1": recall_at_k(
            retrieved, relevant, 1
        ),
        "recall@3": recall_at_k(
            retrieved, relevant, 3
        ),
        "recall@5": recall_at_k(
            retrieved, relevant, 5
        ),
        "recall@10": recall_at_k(
            retrieved, relevant, 10
        ),

        "hit@1": hit_at_k(
            retrieved, relevant, 1
        ),
        "hit@3": hit_at_k(
            retrieved, relevant, 3
        ),
        "hit@5": hit_at_k(
            retrieved, relevant, 5
        ),
        "hit@10": hit_at_k(
            retrieved, relevant, 10
        ),

        "rr": reciprocal_rank(
            retrieved,
            relevant
        ),

        "latency_ms": latency * 1000
    })


chunk_results = pd.DataFrame(rows)

In [58]:
print("=== Chunk + Mean Pooling ===")

print(
    f"Recall@1 : "
    f"{chunk_results['recall@1'].mean():.4f}"
)

print(
    f"Recall@3 : "
    f"{chunk_results['recall@3'].mean():.4f}"
)

print(
    f"Recall@5 : "
    f"{chunk_results['recall@5'].mean():.4f}"
)

print(
    f"Recall@10: "
    f"{chunk_results['recall@10'].mean():.4f}"
)

print()

print(
    f"Hit@1    : "
    f"{chunk_results['hit@1'].mean():.4f}"
)

print(
    f"Hit@5    : "
    f"{chunk_results['hit@5'].mean():.4f}"
)

print(
    f"Hit@10   : "
    f"{chunk_results['hit@10'].mean():.4f}"
)

print()

print(
    f"MRR      : "
    f"{chunk_results['rr'].mean():.4f}"
)

print(
    f"Avg latency: "
    f"{chunk_results['latency_ms'].mean():.2f} ms"
)

=== Chunk + Mean Pooling ===
Recall@1 : 0.3900
Recall@3 : 0.4440
Recall@5 : 0.4600
Recall@10: 0.5300

Hit@1    : 0.4800
Hit@5    : 0.6600
Hit@10   : 0.7600

MRR      : 0.5667
Avg latency: 19.85 ms


In [59]:
chunk_by_type = (
    chunk_results
    .groupby("query_type")
    [
        [
            "recall@1",
            "recall@3",
            "recall@5",
            "recall@10",
            "hit@1",
            "hit@5",
            "hit@10",
            "rr"
        ]
    ]
    .mean()
    .round(3)
)

chunk_by_type

,recall@1,recall@3,recall@5,recall@10,hit@1,hit@5,hit@10,rr
query_type,,,,,,,,
content_qa,0.933,0.933,0.933,0.933,0.933,0.933,0.933,0.933
cross_story,0.000,0.250,0.250,0.375,0.000,0.500,0.750,0.292
exact_lookup,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
exact_title,0.200,0.400,0.400,0.600,0.200,0.400,0.600,0.333
long_story_qa,0.800,0.800,0.800,0.800,0.800,0.800,0.800,0.800
metadata,0.038,0.075,0.112,0.225,0.375,0.625,0.750,0.476
semantic_discovery,0.020,0.060,0.110,0.220,0.200,0.600,0.800,0.369


In [40]:
metrics = [
    "recall@1",
    "recall@3",
    "recall@5",
    "recall@10",
    "hit@1",
    "hit@5",
    "hit@10",
    "rr"
]

comparison = pd.DataFrame({
    "Truncated": results[metrics].mean(),
    "Chunk_Mean_Pooling": chunk_results[metrics].mean()
})

comparison["Difference"] = (
    comparison["Chunk_Mean_Pooling"]
    - comparison["Truncated"]
)

comparison.round(4)

,Truncated,Chunk_Mean_Pooling,Difference
recall@1,0.214,0.3800,0.1660
recall@3,0.294,0.4400,0.1460
recall@5,0.382,0.4760,0.0940
recall@10,0.414,0.5180,0.1040
hit@1,0.340,0.4800,0.1400
hit@5,0.580,0.6800,0.1000
hit@10,0.580,0.7400,0.1600
rr,0.418,0.5623,0.1443


In [42]:
type_comparison = (
    results
    .groupby("query_type")[metrics]
    .mean()
    .join(
        chunk_results
        .groupby("query_type")[metrics]
        .mean(),
        lsuffix="_truncated",
        rsuffix="_chunk_pool"
    )
)



In [43]:
important_types = [
    "semantic_discovery",
    "content_qa",
    "long_story_qa",
    "cross_story"
]

for query_type in important_types:

    print(f"\n=== {query_type} ===")

    old = results[
        results["query_type"] == query_type
    ]

    new = chunk_results[
        chunk_results["query_type"] == query_type
    ]

    print(
        f"Hit@5: "
        f"{old['hit@5'].mean():.3f}"
        f" → "
        f"{new['hit@5'].mean():.3f}"
    )

    print(
        f"Recall@10: "
        f"{old['recall@10'].mean():.3f}"
        f" → "
        f"{new['recall@10'].mean():.3f}"
    )

    print(
        f"MRR: "
        f"{old['rr'].mean():.3f}"
        f" → "
        f"{new['rr'].mean():.3f}"
    )


=== semantic_discovery ===
Hit@5: 0.700 → 0.600
Recall@10: 0.250 → 0.220
MRR: 0.353 → 0.353

=== content_qa ===
Hit@5: 0.733 → 0.933
Recall@10: 0.733 → 0.933
MRR: 0.547 → 0.933

=== long_story_qa ===
Hit@5: 0.800 → 0.800
Recall@10: 0.800 → 0.800
MRR: 0.667 → 0.800

=== cross_story ===
Hit@5: 0.000 → 0.500
Recall@10: 0.000 → 0.250
MRR: 0.000 → 0.375


# Experiment 3 — Summary-Based Story Representation

In [15]:
import numpy as np
import pandas as pd

RANDOM_SEED = 42

# Story IDs that appear in retrieval-sensitive benchmark queries
retrieval_types = [
    "semantic_discovery",
    "content_qa",
    "long_story_qa",
    "cross_story"
]

retrieval_benchmark = benchmark[
    benchmark["query_type"].isin(retrieval_types)
].copy()

benchmark_story_ids = set()

for value in retrieval_benchmark["relevant_story_ids"]:
    benchmark_story_ids.update(
        parse_relevant_ids(value)
    )

len(benchmark_story_ids)

111

In [16]:
df["length_group"] = pd.cut(
    df["token_count_minilm"],
    bins=[0, 700, 1500, float("inf")],
    labels=["short", "medium", "long"]
)

df["length_group"].value_counts()

length_group
medium    689
long      204
short     107
Name: count, dtype: int64

In [17]:
candidate_stories = df[
    df["id"].isin(benchmark_story_ids)
].copy()

candidate_stories[
    ["id", "title", "genre", "token_count_minilm", "length_group"]
].head()

,id,title,genre,token_count_minilm,length_group
0,457580,The Chronicles of the Cosmic Rift,Science Fiction,1661,long
1,297904,Eldoria's Enchanted Whispers,Fantasy,1335,medium
2,620436,Echoes of Whispered Shadows,Mystery,931,medium
5,313009,Sands' Timeless Odyssey,Historical Fiction,918,medium
7,523790,Willowbrook Manor's Ghostly Echoes,Horror,1058,medium


In [18]:
target_counts = {
    "short": 5,
    "medium": 8,
    "long": 7
}

selected_parts = []

for group, n in target_counts.items():

    group_df = candidate_stories[
        candidate_stories["length_group"] == group
    ]

    n_sample = min(n, len(group_df))

    selected_parts.append(
        group_df.sample(
            n=n_sample,
            random_state=RANDOM_SEED
        )
    )

pilot_stories = pd.concat(
    selected_parts
).drop_duplicates("id")

In [19]:
len(pilot_stories)

20

In [20]:
if len(pilot_stories) < 20:

    remaining = candidate_stories[
        ~candidate_stories["id"].isin(
            pilot_stories["id"]
        )
    ]

    needed = 20 - len(pilot_stories)

    extra = remaining.sample(
        n=min(needed, len(remaining)),
        random_state=RANDOM_SEED
    )

    pilot_stories = pd.concat([
        pilot_stories,
        extra
    ])

pilot_stories = pilot_stories.reset_index(drop=True)

print("Pilot size:", len(pilot_stories))

display(
    pilot_stories[
        [
            "id",
            "title",
            "genre",
            "token_count_minilm",
            "length_group"
        ]
    ]
)

Pilot size: 20


,id,title,genre,token_count_minilm,length_group
0,506286,The Invisible Hand,Spy Fiction,210,short
1,173873,The House of Shadows,Gothic,287,short
2,308506,The Chronicles of Eldareth: The Quest for the ...,Fantasy,146,short
3,313112,The Starfarers Dilemma,Space Opera,198,short
4,552359,The Dark Corner,Horror,201,short
5,979918,The Interstellar Odyssey,Science Fiction,1278,medium
6,297904,Eldoria's Enchanted Whispers,Fantasy,1335,medium
7,251638,The Enigma of Emerald Hill,Mystery,1319,medium
8,488842,The Chronicles of Time: A Journey Through Ages,Time Travel,897,medium
9,377725,The Awakening of A.I.,Artificial Intelligence,1315,medium


In [21]:
print("Length distribution:")
display(
    pilot_stories["length_group"]
    .value_counts()
)

print("\nGenres:")
print(pilot_stories["genre"].nunique())

Length distribution:


length_group
medium    8
long      7
short     5
Name: count, dtype: int64


Genres:
12


In [13]:
def benchmark_usage(story_id):

    used_in = []

    for _, row in retrieval_benchmark.iterrows():

        relevant_ids = parse_relevant_ids(
            row["relevant_story_ids"]
        )

        if story_id in relevant_ids:
            used_in.append(row["query_type"])

    return sorted(set(used_in))


In [ ]:


pilot_stories["benchmark_usage"] = (
    pilot_stories["id"]
    .apply(benchmark_usage)
)

display(
    pilot_stories[
        [
            "id",
            "title",
            "length_group",
            "benchmark_usage"
        ]
    ]
)

,id,title,length_group,benchmark_usage
0,506286,The Invisible Hand,short,[semantic_discovery]
1,173873,The House of Shadows,short,"[content_qa, cross_story]"
2,308506,The Chronicles of Eldareth: The Quest for the ...,short,[semantic_discovery]
3,313112,The Starfarers Dilemma,short,[semantic_discovery]
4,552359,The Dark Corner,short,[semantic_discovery]
5,979918,The Interstellar Odyssey,medium,[semantic_discovery]
6,297904,Eldoria's Enchanted Whispers,medium,[semantic_discovery]
7,251638,The Enigma of Emerald Hill,medium,[semantic_discovery]
8,488842,The Chronicles of Time: A Journey Through Ages,medium,[semantic_discovery]
9,377725,The Awakening of A.I.,medium,[semantic_discovery]


## Groq setup

In [14]:
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv("../.env")

client = Groq(
    api_key= os.environ.get("GROQ_API_KEY")
)

SUMMARY_MODEL = "openai/gpt-oss-120b"

In [15]:
client

In [16]:
SUMMARY_SYSTEM_PROMPT = """
You create retrieval-oriented summaries for a story search system.

Summarize the provided story into one dense, self-contained paragraph.

Preserve:
- main characters and their roles
- important relationships
- setting
- main plot and conflict
- major events, discoveries, and decisions
- important objects, locations, organizations, and technologies
- central themes and concepts
- important ending events

Rules:
- Use only information explicitly present in the story.
- Do not invent unsupported details.
- Preserve specific names and distinctive events.
- Do not add commentary or opinions.
- Do not mention that this is a summary.
- Prioritize distinctive information useful for identifying the story.
- Be concise. Prefer roughly 100-140 words for longer stories.
- Short stories may use fewer words when appropriate.
- Do not count words or explain your reasoning.
"""

In [30]:
def build_summary_prompt(row):
    return f"""
Title: {row['title']}
Genre: {row['genre']}

STORY:
{row['story']}
"""

### Testing with one story

In [36]:
def generate_summary(row):

    response = client.chat.completions.create(
        model=SUMMARY_MODEL,

        messages=[
            {
                "role": "system",
                "content": SUMMARY_SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": build_summary_prompt(row)
            }
        ],

        temperature=0.0,
        reasoning_effort="low",
        max_completion_tokens=600
    )

    choice = response.choices[0]

    summary = (choice.message.content or "").strip()

    reasoning_tokens = (
        response.usage.completion_tokens_details.reasoning_tokens
        if response.usage.completion_tokens_details
        else None
    )

    if choice.finish_reason != "stop":
        print(
            f"WARNING: finish_reason={choice.finish_reason}"
        )

    if not summary:
        raise ValueError(
            "Model returned empty summary. "
            f"finish_reason={choice.finish_reason}, "
            f"reasoning_tokens={reasoning_tokens}"
        )

    return summary

In [46]:
def compress_summary(summary):

    response = client.chat.completions.create(
        model=SUMMARY_MODEL,
        messages=[
            {
                "role": "system",
                "content": """
Compress the provided story summary for semantic retrieval.

Preserve:
- character names
- setting
- central conflict
- distinctive events
- important entities
- ending/resolution

Remove secondary details and redundant wording.

Use only information already present in the summary.
Return only one concise paragraph.
Do not explain your reasoning.
"""
            },
            {
                "role": "user",
                "content": summary
            }
        ],
        temperature=0.0,
        reasoning_effort="low",
        max_completion_tokens=400
    )

    compressed = (
        response.choices[0]
        .message.content or ""
    ).strip()

    if not compressed:
        raise ValueError(
            "Compression returned empty content."
        )

    return compressed

In [37]:
test_story = pilot_stories.iloc[0]

test_summary = generate_summary(test_story)

summary_tokens = len(
    tokenizer(
        test_summary,
        add_special_tokens=True,
        truncation=False
    )["input_ids"]
)

print(test_summary)
print()
print("MiniLM tokens:", summary_tokens)

In Berlin, the master spy known as “The Invisible Hand” is tasked with identifying a mole inside the German intelligence agency; while tailing a suspect he discovers a mysterious package left outside the agency’s headquarters, which a gust of wind opens to reveal a coded message sent by the mole as a test, and after painstakingly decoding it he learns the mole’s location, confronts the traitor in a fierce struggle, defeats him and takes him into custody, thereby restoring peace to the agency.

MiniLM tokens: 103


In [38]:
summary_tokens = len(
    tokenizer(
        test_summary,
        add_special_tokens=True,
        truncation=False
    )["input_ids"]
)

print("Summary MiniLM tokens:", summary_tokens)
print("MiniLM max:", model.max_seq_length)

Summary MiniLM tokens: 103
MiniLM max: 256


In [39]:
from pathlib import Path

SUMMARY_PATH = Path(
    "../data/processed/pilot_story_summaries.csv"
)

SUMMARY_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

In [47]:
if SUMMARY_PATH.exists():

    summaries_df = pd.read_csv(
        SUMMARY_PATH
    )

    completed_ids = set(
        summaries_df["id"].tolist()
    )

else:

    summaries_df = pd.DataFrame()

    completed_ids = set()

print(
    "Already completed:",
    len(completed_ids)
)

Already completed: 0


In [48]:
records = (
    summaries_df.to_dict("records")
    if len(summaries_df)
    else []
)

for i, row in pilot_stories.iterrows():

    story_id = row["id"]

    # --------------------------------------------------
    # Skip stories already summarized
    # --------------------------------------------------
    if story_id in completed_ids:
        print(f"Skipping {story_id} - already completed")
        continue

    try:
        print(
            f"[{i + 1}/{len(pilot_stories)}] "
            f"Summarizing: {row['title']}"
        )

        # --------------------------------------------------
        # Generate initial summary
        # --------------------------------------------------
        start = time.perf_counter()

        summary = generate_summary(row)

        initial_summary_tokens = len(
            tokenizer(
                summary,
                add_special_tokens=True,
                truncation=False
            )["input_ids"]
        )

        print(
            f"   Initial summary tokens: "
            f"{initial_summary_tokens}"
        )

        # --------------------------------------------------
        # Compress if summary exceeds MiniLM limit
        # --------------------------------------------------
        was_compressed = False

        if initial_summary_tokens > model.max_seq_length:

            print(
                f"   ⚠ Summary exceeds MiniLM limit "
                f"({initial_summary_tokens} > "
                f"{model.max_seq_length})"
            )

            print("   ↳ Compressing summary...")

            summary = compress_summary(summary)

            was_compressed = True

        # --------------------------------------------------
        # Calculate FINAL summary token count
        # --------------------------------------------------
        summary_tokens = len(
            tokenizer(
                summary,
                add_special_tokens=True,
                truncation=False
            )["input_ids"]
        )

        # --------------------------------------------------
        # Final validation
        # --------------------------------------------------
        if summary_tokens > model.max_seq_length:
            raise ValueError(
                f"Summary still exceeds MiniLM limit "
                f"after compression: "
                f"{summary_tokens} > "
                f"{model.max_seq_length}"
            )

        if summary_tokens < 40:
            print(
                f"   ⚠ WARNING: Summary may be too short "
                f"({summary_tokens} tokens)"
            )

        # --------------------------------------------------
        # Total generation + optional compression time
        # --------------------------------------------------
        elapsed = time.perf_counter() - start

        # --------------------------------------------------
        # Compression statistics
        # --------------------------------------------------
        original_tokens = int(
            row["token_count_minilm"]
        )

        compression_ratio = (
            summary_tokens / original_tokens
        )

        # --------------------------------------------------
        # Create record
        # --------------------------------------------------
        record = {
            "id": story_id,
            "title": row["title"],
            "genre": row["genre"],
            "length_group": str(
                row["length_group"]
            ),

            "original_tokens": original_tokens,

            "summary": summary,

            "initial_summary_tokens": (
                initial_summary_tokens
            ),

            "summary_tokens": summary_tokens,

            "was_compressed": was_compressed,

            "compression_ratio": (
                compression_ratio
            ),

            "generation_time_sec": round(
                elapsed,
                3
            ),

            "summary_model": SUMMARY_MODEL,
        }

        # --------------------------------------------------
        # Save immediately
        # --------------------------------------------------
        records.append(record)

        pd.DataFrame(records).to_csv(
            SUMMARY_PATH,
            index=False
        )

        completed_ids.add(story_id)

        # --------------------------------------------------
        # Progress info
        # --------------------------------------------------
        print(
            f"   ✓ Final summary tokens: "
            f"{summary_tokens}"
        )

        print(
            f"   ✓ Original tokens: "
            f"{original_tokens}"
        )

        print(
            f"   ✓ Compression ratio: "
            f"{compression_ratio:.2%}"
        )

        print(
            f"   ✓ Extra compression used: "
            f"{was_compressed}"
        )

        print(
            f"   ✓ Total generation time: "
            f"{elapsed:.2f}s"
        )

        print("-" * 70)

        # --------------------------------------------------
        # Gentle pacing for Groq API
        # --------------------------------------------------
        time.sleep(2)

    except Exception as e:

        print(
            f"❌ FAILED story {story_id}: "
            f"{type(e).__name__}: {e}"
        )

        print(
            "Previous summaries are already saved. "
            "Waiting before continuing..."
        )

        time.sleep(10)

[1/20] Summarizing: The Invisible Hand
   Initial summary tokens: 125
   ✓ Final summary tokens: 125
   ✓ Original tokens: 210
   ✓ Compression ratio: 59.52%
   ✓ Extra compression used: False
   ✓ Total generation time: 1.31s
----------------------------------------------------------------------
[2/20] Summarizing: The House of Shadows
   Initial summary tokens: 148
   ✓ Final summary tokens: 148
   ✓ Original tokens: 287
   ✓ Compression ratio: 51.57%
   ✓ Extra compression used: False
   ✓ Total generation time: 0.64s
----------------------------------------------------------------------
[3/20] Summarizing: The Chronicles of Eldareth: The Quest for the Lost Orb<|im_end|
   Initial summary tokens: 16
   ⚠ WARNING: Summary may be too short (16 tokens)
   ✓ Final summary tokens: 16
   ✓ Original tokens: 146
   ✓ Compression ratio: 10.96%
   ✓ Extra compression used: False
   ✓ Total generation time: 0.51s
----------------------------------------------------------------------
[4/20] Sum

In [52]:
summaries_df = pd.read_csv(SUMMARY_PATH)

print("Total summaries:", len(summaries_df))

print(
    "Compressed:",
    summaries_df["was_compressed"].sum()
)

print(
    "Over MiniLM limit:",
    (
        summaries_df["summary_tokens"]
        > model.max_seq_length
    ).sum()
)

display(
    summaries_df[
        [
            "original_tokens",
            "initial_summary_tokens",
            "summary_tokens",
            "compression_ratio",
            "generation_time_sec"
        ]
    ].describe(
        percentiles=[
            0.5,
            0.75,
            0.9,
            0.95
        ]
    )
)

Total summaries: 20
Compressed: 1
Over MiniLM limit: 0


,original_tokens,initial_summary_tokens,summary_tokens,compression_ratio,generation_time_sec
count,20.000000,20.000000,20.000000,20.000000,20.000000
mean,1254.000000,187.900000,183.850000,0.234619,8.139050
std,814.240169,57.028986,53.350159,0.194604,8.151115
min,146.000000,16.000000,16.000000,0.066691,0.421000
50%,1317.000000,197.000000,194.500000,0.150351,5.934000
75%,1654.750000,229.750000,224.500000,0.223529,14.063750
90%,1977.200000,235.000000,233.200000,0.592360,18.533700
95%,2803.100000,236.950000,235.000000,0.600325,23.107050
max,3071.000000,274.000000,235.000000,0.696970,24.210000


In [49]:
summaries_df = pd.read_csv(
    SUMMARY_PATH
)

summaries_df[
    [
        "original_tokens",
        "summary_tokens",
        "generation_time_sec"
    ]
].describe()

,original_tokens,summary_tokens,generation_time_sec
count,20.000000,20.000000,20.000000
mean,1254.000000,183.850000,8.139050
std,814.240169,53.350159,8.151115
min,146.000000,16.000000,0.421000
25%,719.000000,176.500000,0.749750
50%,1317.000000,194.500000,5.934000
75%,1654.750000,224.500000,14.063750
max,3071.000000,235.000000,24.210000


In [50]:
summaries_df["compression_ratio"] = (
    summaries_df["summary_tokens"]
    /
    summaries_df["original_tokens"]
)

summaries_df[
    [
        "original_tokens",
        "summary_tokens",
        "compression_ratio"
    ]
].describe()

,original_tokens,summary_tokens,compression_ratio
count,20.000000,20.000000,20.000000
mean,1254.000000,183.850000,0.234619
std,814.240169,53.350159,0.194604
min,146.000000,16.000000,0.066691
25%,719.000000,176.500000,0.121541
50%,1317.000000,194.500000,0.150351
75%,1654.750000,224.500000,0.223529
max,3071.000000,235.000000,0.696970


In [51]:
too_long = summaries_df[
    summaries_df["summary_tokens"]
    > model.max_seq_length
]

print(
    "Summaries exceeding embedding limit:",
    len(too_long)
)

Summaries exceeding embedding limit: 0


### An initial 20-story pilot found that 35% of generated summaries exceeded the embedding model's 256-token context window. The summarization pipeline was therefore revised with concise retrieval-oriented generation and token-aware fallback compression. In the second pilot, all 20 summaries fit within the embedding context, with only 1/20 requiring additional compression.

In [53]:
# All 1000 stories
print("Total stories:", len(df))

# Already generated summaries
summaries_df = pd.read_csv(SUMMARY_PATH)

completed_ids = set(
    summaries_df["id"].astype(str)
)

# Make ID type consistent
df["id"] = df["id"].astype(str)

remaining_df = df[
    ~df["id"].isin(completed_ids)
].copy()

print("Completed:", len(completed_ids))
print("Remaining:", len(remaining_df))

Total stories: 1000
Completed: 20
Remaining: 980


### We investigated retrieval-oriented LLM-generated story summaries as an alternative global representation. A 20-story stratified pilot was performed, including short, medium, and long stories from the retrieval benchmark. Token-aware validation ensured all summaries fit within the embedding model's context window. Due to the additional offline preprocessing cost and assessment time constraints, the production baseline used chunk-derived story representations, while summary-based indexing remains an evaluated extension.

# Experiment 4 — Hierarchical Story + Evidence Retrieval

In [60]:
SPECIAL_TOKENS = tokenizer.num_special_tokens_to_add(
    pair=False
)

CHUNK_SIZE = (
    model.max_seq_length
    - SPECIAL_TOKENS
)

CHUNK_OVERLAP = 50

CHUNK_STRIDE = (
    CHUNK_SIZE
    - CHUNK_OVERLAP
)

print("Model max length:", model.max_seq_length)
print("Special tokens:", SPECIAL_TOKENS)
print("Chunk size:", CHUNK_SIZE)
print("Chunk overlap:", CHUNK_OVERLAP)
print("Chunk stride:", CHUNK_STRIDE)

Model max length: 256
Special tokens: 2
Chunk size: 254
Chunk overlap: 50
Chunk stride: 204


In [61]:
def chunk_story(
    text,
    tokenizer,
    chunk_size=CHUNK_SIZE,
    overlap=CHUNK_OVERLAP
):
    token_ids = tokenizer.encode(
        text,
        add_special_tokens=False
    )

    stride = chunk_size - overlap

    chunks = []

    chunk_id = 0

    for start in range(
        0,
        len(token_ids),
        stride
    ):
        end = min(
            start + chunk_size,
            len(token_ids)
        )

        chunk_ids = token_ids[start:end]

        chunk_text = tokenizer.decode(
            chunk_ids,
            skip_special_tokens=True
        )

        chunks.append({
            "chunk_id": chunk_id,
            "start_token": start,
            "end_token": end,
            "text": chunk_text
        })

        chunk_id += 1

        # Story completely covered
        if end >= len(token_ids):
            break

    return chunks

In [62]:
chunk_records = []

for _, row in df.iterrows():

    chunks = chunk_story(
        row["story"],
        tokenizer
    )

    for chunk in chunks:

        chunk_records.append({
            "story_id": str(row["id"]),
            "title": row["title"],
            "genre": row["genre"],
            "chunk_id": chunk["chunk_id"],
            "start_token": chunk["start_token"],
            "end_token": chunk["end_token"],
            "text": chunk["text"]
        })

chunks_df = pd.DataFrame(chunk_records)

print("Stories:", len(df))
print("Chunks:", len(chunks_df))

display(chunks_df.head())

Stories: 1000
Chunks: 6131


,story_id,title,genre,chunk_id,start_token,end_token,text
0,457580,The Chronicles of the Cosmic Rift,Science Fiction,0,0,254,"in the year 2250, earth had made significant s..."
1,457580,The Chronicles of the Cosmic Rift,Science Fiction,1,204,458,"of numerous crea expeditions ; dr. evelyn "" ev..."
2,457580,The Chronicles of the Cosmic Rift,Science Fiction,2,408,662,"verge of being torn apart, dr. simmons managed..."
3,457580,The Chronicles of the Cosmic Rift,Science Fiction,3,612,866,each passing moment. as the crew of the excels...
4,457580,The Chronicles of the Cosmic Rift,Science Fiction,4,816,1070,a hero who would rise to defeat the enigma and...


In [63]:
chunks_per_story = (
    chunks_df
    .groupby("story_id")
    .size()
)

display(
    chunks_per_story.describe(
        percentiles=[0.5, 0.9, 0.95, 0.99]
    )
)

count    1000.000000
mean        6.131000
std         2.577301
min         1.000000
50%         6.000000
90%         9.000000
95%        10.000000
99%        14.000000
max        18.000000
dtype: float64

In [64]:
all_chunk_texts = (
    chunks_df["text"]
    .tolist()
)

start = time.perf_counter()

chunk_embeddings = model.encode(
    all_chunk_texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

chunk_encoding_time = (
    time.perf_counter() - start
)

print(
    "Chunk embeddings shape:",
    chunk_embeddings.shape
)

print(
    f"Encoding time: "
    f"{chunk_encoding_time:.2f}s"
)

Batches: 100%|██████████| 96/96 [02:51<00:00,  1.79s/it]

Chunk embeddings shape: (6131, 384)
Encoding time: 171.70s


In [65]:
story_embeddings_hierarchical = []
story_ids_hierarchical = []

for story_id, group in chunks_df.groupby(
    "story_id",
    sort=False
):

    indices = group.index.to_numpy()

    embeddings = chunk_embeddings[
        indices
    ]

    pooled = embeddings.mean(axis=0)

    pooled = pooled / np.linalg.norm(
        pooled
    )

    story_embeddings_hierarchical.append(
        pooled
    )

    story_ids_hierarchical.append(
        story_id
    )

story_embeddings_hierarchical = np.vstack(
    story_embeddings_hierarchical
)

story_ids_hierarchical = np.array(
    story_ids_hierarchical
)

print(
    story_embeddings_hierarchical.shape
)

(1000, 384)


In [66]:
def retrieve_stories_hierarchical(
    query,
    top_k=5
):

    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    scores = (
        story_embeddings_hierarchical
        @ query_embedding
    )

    top_indices = np.argsort(
        scores
    )[::-1][:top_k]

    results_list = []

    for idx in top_indices:

        story_id = (
            story_ids_hierarchical[idx]
        )

        story_row = df[
            df["id"].astype(str)
            == story_id
        ].iloc[0]

        results_list.append({
            "story_id": story_id,
            "title": story_row["title"],
            "genre": story_row["genre"],
            "score": scores[idx]
        })

    return pd.DataFrame(
        results_list
    )

In [67]:
retrieve_stories_hierarchical(
    "a story involving time travel",
    top_k=5
)

,story_id,title,genre,score
0,687889,The Time Travelers Chronicles: The Adventures ...,Time Travel,0.608517
1,390933,The Timeless Clock,Alternate History,0.597866
2,549313,The Chronicles of the Time-Shifted,Alternate Reality,0.548625
3,461467,The Time Travel Chronicles of the Lost Timekee...,Time Travel,0.543328
4,441294,The Time Travelers Quest,Soft Science Fiction,0.532163


In [68]:
def retrieve_evidence(
    query,
    candidate_story_ids,
    top_k=3
):

    candidate_story_ids = {
        str(x)
        for x in candidate_story_ids
    }

    mask = (
        chunks_df["story_id"]
        .isin(candidate_story_ids)
        .to_numpy()
    )

    candidate_indices = np.where(
        mask
    )[0]

    candidate_embeddings = (
        chunk_embeddings[
            candidate_indices
        ]
    )

    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    scores = (
        candidate_embeddings
        @ query_embedding
    )

    local_top = np.argsort(
        scores
    )[::-1][:top_k]

    global_indices = (
        candidate_indices[
            local_top
        ]
    )

    results = (
        chunks_df
        .iloc[global_indices]
        .copy()
    )

    results["score"] = (
        scores[local_top]
    )

    return results[
        [
            "story_id",
            "title",
            "chunk_id",
            "start_token",
            "end_token",
            "text",
            "score"
        ]
    ]

In [69]:
content_queries = benchmark[
    benchmark["query_type"]
    == "content_qa"
]

test_query = (
    content_queries.iloc[0]["query"]
)

print("QUERY:")
print(test_query)

QUERY:
In "The Chronicles of the Celestial Spoon", what is the name of the spoon that can create horrifying and inedible dishes?


In [30]:
story_results = (
    retrieve_stories_hierarchical(
        test_query,
        top_k=5
    )
)

display(story_results)


,story_id,title,genre,score
0,987253,The Chronicles of the Celestial Spoon,Supernatural Comedy,0.724890
1,157222,The Chronicles of Lunas Café: A Supernatural C...,Supernatural Comedy,0.348320
2,491242,The Legacy of the Celestial Sword,Mythic Fiction,0.300165
3,305099,The Chronicles of the Curious Café,Philosophical Comedy,0.299372
4,171418,The Chronicles of the Giggling Grubs,Satire,0.295803


In [31]:
evidence_results = retrieve_evidence(
    query=test_query,
    candidate_story_ids=(
        story_results["story_id"]
        .tolist()
    ),
    top_k=3
)

display(
    evidence_results[
        [
            "story_id",
            "title",
            "chunk_id",
            "score",
            "text"
        ]
    ]
)

,story_id,title,chunk_id,score,text
1900,987253,The Chronicles of the Celestial Spoon,1,0.687430,"celestial spoon. according to the book, the ce..."
1899,987253,The Chronicles of the Celestial Spoon,0,0.681058,"once upon a time, in the small town of celesti..."
1902,987253,The Chronicles of the Celestial Spoon,3,0.652204,"the battle raged on, amelia, the baker, manage..."


In [32]:
test_row = content_queries.iloc[0]

gold_story_ids = parse_relevant_ids(
    test_row["relevant_story_ids"]
)

oracle_evidence = retrieve_evidence(
    query=test_row["query"],
    candidate_story_ids=gold_story_ids,
    top_k=3
)

print("QUERY:")
print(test_row["query"])

print("\nGOLD STORIES:")
print(gold_story_ids)

display(
    oracle_evidence[
        [
            "title",
            "chunk_id",
            "score",
            "text"
        ]
    ]
)

QUERY:
In "The Chronicles of the Celestial Spoon", what is the name of the spoon that can create horrifying and inedible dishes?

GOLD STORIES:
[987253]


,title,chunk_id,score,text
1900,The Chronicles of the Celestial Spoon,1,0.687430,"celestial spoon. according to the book, the ce..."
1899,The Chronicles of the Celestial Spoon,0,0.681058,"once upon a time, in the small town of celesti..."
1902,The Chronicles of the Celestial Spoon,3,0.652204,"the battle raged on, amelia, the baker, manage..."


In [70]:
hierarchical_results = []

for _, row in benchmark.iterrows():

    query = row["query"]

    relevant_ids = {
        str(x)
        for x in parse_relevant_ids(
            row["relevant_story_ids"]
        )
    }

    start = time.perf_counter()

    retrieved = retrieve_stories_hierarchical(
        query,
        top_k=10
    )

    latency_ms = (
        time.perf_counter() - start
    ) * 1000

    retrieved_ids = (
        retrieved["story_id"]
        .astype(str)
        .tolist()
    )

    hierarchical_results.append({
        "query_id": row["query_id"],
        "query_type": row["query_type"],

        "recall@1": recall_at_k(
            retrieved_ids,
            relevant_ids,
            1
        ),

        "recall@3": recall_at_k(
            retrieved_ids,
            relevant_ids,
            3
        ),

        "recall@5": recall_at_k(
            retrieved_ids,
            relevant_ids,
            5
        ),

        "recall@10": recall_at_k(
            retrieved_ids,
            relevant_ids,
            10
        ),

        "hit@1": hit_at_k(
            retrieved_ids,
            relevant_ids,
            1
        ),

        "hit@5": hit_at_k(
            retrieved_ids,
            relevant_ids,
            5
        ),

        "hit@10": hit_at_k(
            retrieved_ids,
            relevant_ids,
            10
        ),

        "rr": reciprocal_rank(
            retrieved_ids,
            relevant_ids
        ),

        "latency_ms": latency_ms
    })


hierarchical_results = pd.DataFrame(
    hierarchical_results
)

In [71]:
metrics = [
    "recall@1",
    "recall@3",
    "recall@5",
    "recall@10",
    "hit@1",
    "hit@5",
    "hit@10",
    "rr"
]

comparison = pd.DataFrame({
    "Truncated": (
        results[metrics].mean()
    ),

    "Chunk_No_Overlap": (
        chunk_results[metrics].mean()
    ),

    "Chunk_Overlap_50": (
        hierarchical_results[metrics].mean()
    )
})

comparison.round(4)

,Truncated,Chunk_No_Overlap,Chunk_Overlap_50
recall@1,0.214,0.3900,0.4100
recall@3,0.294,0.4440,0.4260
recall@5,0.382,0.4600,0.4440
recall@10,0.414,0.5300,0.5100
hit@1,0.340,0.4800,0.5000
hit@5,0.580,0.6600,0.6400
hit@10,0.580,0.7600,0.7200
rr,0.418,0.5667,0.5547


In [72]:
important_types = [
    "semantic_discovery",
    "content_qa",
    "long_story_qa",
    "cross_story"
]

hierarchical_by_type = (
    hierarchical_results[
        hierarchical_results["query_type"]
        .isin(important_types)
    ]
    .groupby("query_type")[metrics]
    .mean()
    .round(3)
)

display(hierarchical_by_type)

,recall@1,recall@3,recall@5,recall@10,hit@1,hit@5,hit@10,rr
query_type,,,,,,,,
content_qa,0.933,0.933,0.933,0.933,0.933,0.933,0.933,0.933
cross_story,0.000,0.125,0.125,0.250,0.000,0.250,0.500,0.167
long_story_qa,0.800,0.800,0.800,0.800,0.800,0.800,0.800,0.800
semantic_discovery,0.020,0.050,0.080,0.180,0.200,0.600,0.700,0.330


In [73]:
oracle_evidence = retrieve_evidence(
    query=test_row["query"],
    candidate_story_ids=gold_story_ids,
    top_k=3
)

for rank, (_, chunk) in enumerate(
    oracle_evidence.iterrows(),
    start=1
):
    print("=" * 80)

    print(
        f"RANK {rank} | "
        f"Chunk {chunk['chunk_id']} | "
        f"Score {chunk['score']:.4f}"
    )

    print("=" * 80)

    print(chunk["text"])

    print()

RANK 1 | Chunk 1 | Score 0.6874
celestial spoon. according to the book, the celestial spoon was created by the gods themselves, as a gift to humanity. the spoon was said to have the power to bring people together and to turn even the most ordinary ingredients into the most exquisite dishes. however, the story took a darker turn when benjamin discovered that the celestial spoon was guarded by a powerful sorceress named luna, who had been using the spoon ' s powers for her own evil deeds. she was determined to keep the spoon hidden from the world, so she could continue her wicked schemes. benjamin knew he had to find a way to defeat luna and claim the celestial spoon for himself. he enlisted the help of his newfound friends - a wise old librarian named theodore, a charming baker named amelia, and a mischievous little sprite named sprout - who shared his passion for delicious food. together, the group embarked on a thrilling adventure filled with magical encounters, heart - pounding escap

In [74]:
query = test_row["query"]

all_story_evidence = retrieve_evidence(
    query=query,
    candidate_story_ids=["987253"],
    top_k=20
)

for rank, (_, chunk) in enumerate(
    all_story_evidence.iterrows(),
    start=1
):
    print(
        f"\n{'='*80}\n"
        f"RANK {rank} | "
        f"Chunk {chunk['chunk_id']} | "
        f"Score {chunk['score']:.4f}"
        f"\n{'='*80}"
    )

    print(chunk["text"])


RANK 1 | Chunk 1 | Score 0.6874
celestial spoon. according to the book, the celestial spoon was created by the gods themselves, as a gift to humanity. the spoon was said to have the power to bring people together and to turn even the most ordinary ingredients into the most exquisite dishes. however, the story took a darker turn when benjamin discovered that the celestial spoon was guarded by a powerful sorceress named luna, who had been using the spoon ' s powers for her own evil deeds. she was determined to keep the spoon hidden from the world, so she could continue her wicked schemes. benjamin knew he had to find a way to defeat luna and claim the celestial spoon for himself. he enlisted the help of his newfound friends - a wise old librarian named theodore, a charming baker named amelia, and a mischievous little sprite named sprout - who shared his passion for delicious food. together, the group embarked on a thrilling adventure filled with magical encounters, heart - pounding esca

### Overlapping chunks slightly reduced aggregate story-level retrieval performance compared with non-overlapping mean pooling (MRR 0.555 vs. 0.567), while preserving contextual continuity for downstream evidence retrieval. Therefore, the final architecture separates the two concerns: non-overlapping chunk representations are pooled for story-level retrieval, while overlapping chunks are retained for fine-grained evidence retrieval.